# β-Caryophyllene across HepG2 and UFH-001: gene-effect comparison

**Status:** unexecuted method notebook. Run only after the UFH-001 raw-count notebook and its execution receipt are available. The comparison reads receipt-pinned PyDESeq2 result tables from two public project repositories.

HepG2 (GSE276099) and UFH-001 (GSE125511) differ in cell type, oxygen, dose, time, and vehicle. This notebook measures descriptive gene-level directional agreement within the common tested gene universe. It does not estimate a pooled treatment effect, claim biological replication, infer pathway activity, or demonstrate direct target engagement. UFH-001's two dose contrasts reuse the same three DMSO controls and are not independent experiments.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import io
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
UFH = ROOT / "research/ufh001-bcp-fpkm/results/raw_counts"
UFH_RECEIPT = ROOT / "research/execution-receipts/terputer-ufh001-bcp-raw-counts.json"
receipt = json.loads(UFH_RECEIPT.read_text())
for filename in ("deseq2_bcp20_vs_vehicle.tsv", "deseq2_bcp200_vs_vehicle.tsv"):
    path = UFH / filename
    relative = str(path.relative_to(ROOT))
    assert hashlib.sha256(path.read_bytes()).hexdigest() == receipt["artifacts_written"][relative], relative
OUT = UFH / "cross_context"
OUT.mkdir(parents=True, exist_ok=True)
HEPG2_COMMIT = "76bbc4748d6d0590bea4fa7fb3a3e32c80c74d33"
HEPG2_PATH = "research/hepg2-bcp-rsem/results/raw_counts/deseq2_results.tsv"
HEPG2_SHA256 = "af2559428959ff5b366bf3899d37da1cd1d87d9ddc8f0eb33a19077d685585ef"
url = f"https://raw.githubusercontent.com/Terpedia/terpedia-hepg2-bcp-rsem/{HEPG2_COMMIT}/{HEPG2_PATH}"
with urlopen(url, timeout=60) as response:
    hepg2_bytes = response.read()
assert hashlib.sha256(hepg2_bytes).hexdigest() == HEPG2_SHA256
hepg2 = pd.read_csv(io.BytesIO(hepg2_bytes), sep="\t")
ufh20 = pd.read_csv(UFH / "deseq2_bcp20_vs_vehicle.tsv", sep="\t")
ufh200 = pd.read_csv(UFH / "deseq2_bcp200_vs_vehicle.tsv", sep="\t")
for label, table in (("hepg2", hepg2), ("ufh20", ufh20), ("ufh200", ufh200)):
    assert {"gene_id", "gene_name", "log2FoldChange", "padj"}.issubset(table.columns), label
    assert table.gene_id.notna().all() and table.gene_id.is_unique, label
    assert pd.to_numeric(table.log2FoldChange, errors="coerce").notna().all(), label
print({"hepg2_tested": len(hepg2), "ufh20_tested": len(ufh20), "ufh200_tested": len(ufh200)})


## Prespecified comparison

Join by GENCODE v48 versioned `gene_id`. For each UFH-001 dose, report the common tested universe, Spearman correlation of log2 fold changes, and direction counts among genes with BH adjusted p < 0.05 in **both** contrasts. Both-significant genes are a selection subset; its concordance is descriptive and cannot establish reproducibility. Report the overlapping gene IDs and gene names to make the comparison auditable. No p-value pooling or cross-study FDR is performed.


In [ ]:
def compare(ufh, dose):
    left = hepg2[["gene_id", "gene_name", "log2FoldChange", "padj"]].rename(columns={
        "gene_name": "hepg2_gene_name", "log2FoldChange": "hepg2_log2fc", "padj": "hepg2_padj"})
    right = ufh[["gene_id", "gene_name", "log2FoldChange", "padj"]].rename(columns={
        "gene_name": "ufh_gene_name", "log2FoldChange": "ufh_log2fc", "padj": "ufh_padj"})
    joined = left.merge(right, on="gene_id", validate="one_to_one")
    if joined.empty:
        raise ValueError(f"No common tested genes for {dose}")
    mismatched_names = joined.hepg2_gene_name.notna() & joined.ufh_gene_name.notna() & joined.hepg2_gene_name.ne(joined.ufh_gene_name)
    if mismatched_names.any():
        raise ValueError(f"Gene-name annotation mismatch for {dose}: {joined.loc[mismatched_names, 'gene_id'].tolist()[:5]}")
    joined.insert(1, "gene_name", joined.hepg2_gene_name.combine_first(joined.ufh_gene_name))
    joined = joined.drop(columns=["hepg2_gene_name", "ufh_gene_name"])
    joined["hepg2_fdr05"] = joined.hepg2_padj.lt(0.05)
    joined["ufh_fdr05"] = joined.ufh_padj.lt(0.05)
    joined["both_fdr05"] = joined.hepg2_fdr05 & joined.ufh_fdr05
    joined["same_direction"] = np.sign(joined.hepg2_log2fc).eq(np.sign(joined.ufh_log2fc))
    joined = joined.sort_values("gene_id")
    joined.to_csv(OUT / f"gene_effects_hepg2_vs_ufh_{dose}.tsv", sep="\t", index=False)
    both = joined.loc[joined.both_fdr05]
    return {
        "dose": dose,
        "common_tested_genes": int(len(joined)),
        "hepg2_fdr05_in_common_universe": int(joined.hepg2_fdr05.sum()),
        "ufh_fdr05_in_common_universe": int(joined.ufh_fdr05.sum()),
        "both_fdr05": int(len(both)),
        "both_fdr05_same_direction": int(both.same_direction.sum()),
        "both_fdr05_opposite_direction": int((~both.same_direction).sum()),
        "all_common_spearman_log2fc": float(joined.hepg2_log2fc.corr(joined.ufh_log2fc, method="spearman")),
        "shared_ufh_controls": True,
    }

summaries = [compare(ufh20, "20uM"), compare(ufh200, "200uM")]
summary = {"hepg2_commit": HEPG2_COMMIT, "hepg2_result_sha256": HEPG2_SHA256,
           "comparisons": summaries,
           "limitations": ["Different cell types, oxygen, dose, time, and vehicles",
                           "UFH-001 dose contrasts share the same three controls",
                           "Descriptive cross-study comparison; no pooled inference or biological replication established"]}
(OUT / "comparison_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
pd.DataFrame(summaries).to_csv(OUT / "comparison_summary.tsv", sep="\t", index=False)
pd.DataFrame(summaries)


## Interpretation gate

Describe a gene as showing the same or opposite estimated direction **in these two labeled datasets**. Inspect its per-study effect size and adjusted p value before proposing follow-up. Shared direction alone does not show that the response is reproducible across controlled biological contexts, and an opposite direction does not isolate cell type as the cause because exposure conditions differ. Test context or target dependence with matched dose, time, oxygen, vehicle, viability, and independently replicated cell models, then add direct target or genetic perturbation assays where justified.
